# Saudi Date Fruit - Exploratory Data Analysis

Explore the dataset: class distribution, sample images, image dimensions, and color analysis.

**Prerequisites:** Run `python -m src.data_setup` first to generate the CSV splits.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data Splits

In [ ]:
train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Total: {len(train_df) + len(val_df) + len(test_df)}')
train_df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['variety'].value_counts().sort_index()
    counts.plot(kind='bar', ax=ax, color='#2196F3', edgecolor='white')
    ax.set_title(f'{name} Set ({len(df)} images)', fontsize=14)
    ax.set_xlabel('Variety')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Sample Images per Variety

In [ ]:
varieties = sorted(train_df['variety'].unique())
n_varieties = len(varieties)
samples_per = 3

fig, axes = plt.subplots(n_varieties, samples_per, figsize=(4*samples_per, 3*n_varieties))

for i, variety in enumerate(varieties):
    subset = train_df[train_df['variety'] == variety].sample(n=samples_per, random_state=42)
    for j, (_, row) in enumerate(subset.iterrows()):
        img = Image.open(row['image_path'])
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(variety, fontsize=12, fontweight='bold')

plt.suptitle('Sample Images per Saudi Date Variety', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Image Dimensions

In [ ]:
widths, heights = [], []
for path in train_df['image_path'].sample(min(200, len(train_df)), random_state=42):
    with Image.open(path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

print(f'Width  - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}')
print(f'Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='#2196F3', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[1].hist(heights, bins=30, color='#4CAF50', edgecolor='white')
axes[1].set_title('Image Height Distribution')
plt.tight_layout()
plt.show()

## 5. Average Color per Variety

In [ ]:
avg_colors = {}
for variety in varieties:
    subset = train_df[train_df['variety'] == variety].sample(n=min(20, len(train_df[train_df['variety'] == variety])), random_state=42)
    colors = []
    for path in subset['image_path']:
        img = np.array(Image.open(path).resize((64, 64)))
        colors.append(img.mean(axis=(0,1)))
    avg_colors[variety] = np.mean(colors, axis=0) / 255.0

fig, ax = plt.subplots(figsize=(12, 3))
for i, (variety, color) in enumerate(avg_colors.items()):
    ax.bar(i, 1, color=color, width=0.8)
    ax.text(i, 0.5, variety, ha='center', va='center', fontsize=10, fontweight='bold',
            color='white' if np.mean(color) < 0.5 else 'black')

ax.set_xlim(-0.5, len(varieties)-0.5)
ax.set_ylim(0, 1)
ax.set_title('Average Color per Variety', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()